In [ ]:
import functools
import os
import math
import numpy
import numba
import geopy
import geopy.distance
import scipy.constants as sc

In [ ]:
data_dir = "/home/user/Downloads/HGT_FILES"

# Get elevation

In [ ]:
import numpy as np, os, math, functools


@functools.cache
def load_hgt_file(path: str):
    size = os.path.getsize(path)
    dim = int(math.sqrt(size / 2))
    assert dim**2 * 2 == size
    return np.fromfile(path, ">i2").reshape(dim, dim).astype(np.int16)


@numba.jit
def interpolate_elevation_tile(lat_f: float, lon_f: float, arr: np.ndarray) -> float:
    dim = arr.shape[0]
    # map to array coordinates
    i = (1 - lat_f) * (dim - 1)
    j = lon_f * (dim - 1)

    i0 = int(i)
    j0 = int(j)
    di = i - i0
    dj = j - j0

    # clamp edges
    i1 = min(i0 + 1, dim - 1)
    j1 = min(j0 + 1, dim - 1)

    # bilinear interpolation
    return (
        arr[i0, j0] * (1 - di) * (1 - dj)
        + arr[i1, j0] * di * (1 - dj)
        + arr[i0, j1] * (1 - di) * dj
        + arr[i1, j1] * di * dj
    )


def elevationAt(lat: float, lon: float) -> float:
    lat0 = math.floor(lat)
    lon0 = math.floor(lon)

    ns = "N" if lat0 >= 0 else "S"
    ew = "E" if lon0 >= 0 else "W"
    filename = f"{ns}{abs(lat0):02d}{ew}{abs(lon0):03d}.hgt"

    arr = load_hgt_file(f"{data_dir}/{filename}")

    # local fractional degree within tile
    lat_f = lat - lat0
    lon_f = lon - lon0

    return interpolate_elevation_tile(lat_f, lon_f, arr)

In [ ]:
# Uetliberg.
# https://earthexplorer.usgs.gov/
lat = 47.3495
lon = 8.4914
height = 866

# OpenBurst:
# 47:48:351:352-hd
# 48:49:351:352-hd
# 47:48:352:353-hd
# 48:49:352:353-hd

elevationAt(lat, lon)

In [ ]:
%timeit elevationAt(47.671, 8.5325) # Should be: 560

In [ ]:
# Tests.
# Each row is another test, consisting of (lat, lon) and the expected height.
# The points where chosen randomly using https://www.randomcoords.com/region/europe
# and the correct heights from https://latlongdata.com/elevation/.
test_cases = np.array(
    [
        [47.1380402, 9.5184237, 453.86],
        [46.2313104, 7.3658079, 500.88],
        [45.7391548, 7.3046952, 604.31],
        [47.8007899, 13.0441258, 423.64],
        [49.7452357, 6.6357603, 137.52],
        [50.0973787, 14.5286722, 218.03],
    ]
)

for i in range(test_cases.shape[0]):
    lat, lon, correct_height = test_cases[i, :]
    height = elevationAt(lat, lon)
    diff = np.abs(height - correct_height)
    assert diff < 10

In [ ]:
import pydantic

In [ ]:
class Radar(pydantic.BaseModel):
    id: int
    lat: float
    lon: float
    power: int
    diameter: float
    frequency: float
    pulse_width: float
    cpi_pulses: int
    bandwidth: int
    pfa: float
    min_elevation: float
    max_elevation: float


radar = Radar(
    id=585,
    lat=47.36700085728634,
    lon=8.537724304199216,
    power=20000,
    diameter=2.0,
    frequency=1,
    pulse_width=1,
    cpi_pulses=1,
    bandwidth=1,
    pfa=1e-6,
    min_elevation=-20.0,
    max_elevation=60.0,
)

target_flight_height: float = 1000.0
target_cross_section: float = 2.0

# Radar euqation

In [ ]:
import scipy.constants as sc
from scipy import integrate
from scipy import special

In [ ]:
numba.__version__

In [ ]:
# Taken from OpenBurst.
def marcum_q_fn(v, alpha):
    """integrand function to evaluate the Marcum Q
    function that provides the probability of detection for
    a single pulse out of a quadrature detector.
    """
    return v * np.exp(-(v * v + alpha * alpha) / 2) * special.iv(0, alpha * v)


@functools.cache
def radar_eq_max_dist(
    trans_pwr, antenna_diam, frequency, pulse_width, cpi_pulses, bandwidth, pfa, rcsSM
) -> float:
    """! returns maximal distance given the radar parameters and the target rcs.
    MAX RANGE IS SET TO 400kms, due to the HARD LIMIT in SPLAT! (see MAXPAFES in splatBurst.h)
    """

    ####################### Radar and target values (working) ##################

    c = sc.speed_of_light
    # speed of light
    rf_loss = 12
    # RF system hardware loss (not known for ASR, best guess from chapter 2.12, p.80 Skolnik)
    rcs_start = 10 * np.log10(rcsSM)
    # rcs_start=13;                        # RCS in db
    noise_figure = 1.9
    # Receiver LNA noise figure in dB  (not known for TA, best guess)
    window = 0
    # rectangular for no window, or Hamming (=0)
    equiv_temp = 300
    # equivalent temperature [K] (not known for ASR, best guess)

    ########################################################################################

    ############# ------------- analyze for upto 400 kms
    start_range = 0.001
    # [m] starting at 0 will cause a division by 0 error further down
    # Attention!!!!!! if the snr is very high (that is for low ranges) the besseli function will overflow
    # throwing a segmentation fault (another way to avoid it is to return Pd=1 for snr > e.g. 30dB)
    # this can be avoided by setting the start_range to be a higher value
    stop_range = 400000
    # [m]

    # ------------------------------evaluate  range resolution ---------------

    res = c / (2 * (bandwidth * 1e6))
    if window == 0:
        res = res * 1.44

    # we set resolution manually
    res = 1000  # [m]

    # -----------------------evaluate radar range equation -----------------------------------

    A = np.pi * antenna_diam * antenna_diam / 4
    wavelength = c / (frequency * 1e9)
    antenna_gain = 10 * np.log10(0.6 * 4 * np.pi * A / (wavelength * wavelength))

    four_pi = 10 * np.log10(pow((4 * np.pi), 3))
    pt = 10 * np.log10(trans_pwr)
    lambda_sq = 2 * 10 * np.log10(c / (frequency * 1e9))
    ktb = 10 * np.log10(1.38e-23 * equiv_temp * (bandwidth * 1e6))
    t_bw_gain = 10 * np.log10(pulse_width * bandwidth)
    dop_gain = 10 * np.log10(cpi_pulses)

    ranges = np.arange(start_range, stop_range + 1, res)
    snr = []
    for rng in ranges:
        curr_snr = (
            pt
            + lambda_sq
            + 2 * antenna_gain
            + t_bw_gain
            + dop_gain
            + rcs_start
            - four_pi
            - ktb
            - 40 * np.log10(rng)
            - noise_figure
            - rf_loss
        )
        snr = np.concatenate([snr, [curr_snr]])

    # -----------------------------evaluate Pd function -----------------------

    beta = math.sqrt(-2 * (np.log(pfa)))

    rangel = np.arange(start_range, stop_range + 1, res)

    # lrl = rangel.shape[0]

    jj = 0
    pd = []

    for rng in rangel:
        # avoid segmentation fault in the besseli function for high snr values
        if snr[jj] > 30:
            pd = np.concatenate([pd, [1.0]])
        else:
            alpha = pow(10, (snr[jj] + 3) / 20)
            curr_pd = (
                1
                - integrate.quad(
                    lambda x: marcum_q_fn(x, alpha),
                    0,
                    beta,
                )[0]
            )
            pd = np.concatenate([pd, [curr_pd]])

        jj = jj + 1

    # compute and return max range (attention: all max ranges above this will not be noted by the user)
    retval = 400000

    ii = 0
    for _ in pd:
        if pd[ii] <= 0.8:
            retval = (rangel[ii] + rangel[ii - 1]) / 2
            break
        ii = ii + 1

    # change from m to km
    retval = retval / 1000.0
    return retval


def burstvincentydistance(start_point, dist_km, brng):
    """
    ! returns the destination [lat/lon] point at distance dist_km[km] and at bearing brng[deg] from point pnt[lat/lon];
    default uses the geodesic distance; but also the great-circle distance can be used,
    see: https://geopy.readthedocs.io/en/stable/

    """
    d = geopy.distance.distance(kilometers=dist_km)
    dest = d.destination(point=start_point, bearing=brng)
    return dest


def get_dest_loc_from_dist_and_angle(origin, theta, dist):
    """
    returns destination location (lat/lon) from distance and angle from global origin
    """
    dest = burstvincentydistance(origin, dist, theta)
    return dest, theta

In [ ]:
origin = geopy.Point(radar.lat, radar.lon)
origin_height = elevationAt(radar.lat, radar.lon)
dist_step = 0.5
theta_res = 0.1

In [ ]:
%%timeit
max_dist = radar_eq_max_dist(
    radar.power,
    radar.diameter,
    radar.frequency,
    radar.pulse_width,
    radar.cpi_pulses,
    radar.bandwidth,
    radar.pfa,
    target_cross_section,
)

max_north = burstvincentydistance(origin, max_dist, 0)
max_east = burstvincentydistance(origin, max_dist, 90)
max_south = burstvincentydistance(origin, max_dist, 180)
max_west = burstvincentydistance(origin, max_dist, 270)

In [ ]:
max_dist = radar_eq_max_dist(
    radar.power,
    radar.diameter,
    radar.frequency,
    radar.pulse_width,
    radar.cpi_pulses,
    radar.bandwidth,
    radar.pfa,
    target_cross_section,
)

max_north = burstvincentydistance(origin, max_dist, 0)
max_east = burstvincentydistance(origin, max_dist, 90)
max_south = burstvincentydistance(origin, max_dist, 180)
max_west = burstvincentydistance(origin, max_dist, 270)

lat_min = max_south.latitude
lat_max = max_north.latitude
lon_min = max_west.longitude
lon_max = max_west.longitude

# Line of sight

In [ ]:
class Point(pydantic.BaseModel):
    lat: float
    lon: float
    alt: float

In [ ]:
from pyproj import Transformer


transformer = Transformer.from_crs(
    "epsg:4326",
    "epsg:4978",
    always_xy=True,
)
radar_alt = elevationAt(radar.lat, radar.lon)

In [ ]:
%%timeit
start_xyz = transformer.transform(radar.lon, radar.lat, radar_alt)

In [ ]:
transformer.transform(0, 0, 0)

In [ ]:
from pyproj import Transformer

R_EARTH = 6378137.0  # [m]


def calculate_line_of_sight_endpoint(
    start: Point,
    bearing: float,
    d_max: float,
    delta: float,
) -> float:
    """
    Calculate last point within line of sight for the given resolution.
    """
    transformer = Transformer.from_crs(
        "epsg:4326",
        "epsg:4978",
        always_xy=True,
    )
    # Calculate stop point.
    stop = geopy.distance.distance(meters=d_max).destination(
        point=(start.lat, start.lon, 0),
        bearing=bearing,
    )
    stop.altitude = elevationAt(stop.latitude, stop.longitude)

    # Compute the cosine between the line connecting earth center and start and
    # the line between start and stop.
    start_xyz = transformer.transform(start.lon, start.lat, start.alt)
    stop_xyz = transformer.transform(stop.longitude, stop.latitude, stop.altitude)
    h_start = start.alt + R_EARTH
    h_stop = stop.altitude + R_EARTH
    d_start_stop = np.sqrt(
        np.square(start_xyz[0] - stop_xyz[0])
        + np.square(start_xyz[1] - stop_xyz[1])
        + np.square(start_xyz[2] - stop_xyz[2])
    )

    # Apply cosine law.
    cos_start_earthcenter_stop = (
        np.square(h_start) + np.square(d_start_stop) - np.square(h_stop)
    ) / (2 * d_start_stop * h_start)

    d_visible = 0.0
    for d in np.linspace(delta, d_max, int(d_max // delta)):
        point = geopy.distance.distance(meters=d).destination(
            point=(start.lat, start.lon, start.alt),
            bearing=bearing,
        )
        point.altitude = elevationAt(point.latitude, point.longitude)
        point_xyz = transformer.transform(
            point.longitude, point.latitude, point.altitude
        )
        h_point = point.altitude + R_EARTH
        d_start_point = np.sqrt(
        np.square(start_xyz[0] - point_xyz[0])
        + np.square(start_xyz[1] - point_xyz[1])
        + np.square(start_xyz[2] - point_xyz[2])
    )

        # Apply cosine law.
        cos_start_earthcenter_point = (
            np.square(h_start) + np.square(d_start_point) - np.square(h_point)
        ) / (2 * d_start_point * h_start)

        if cos_start_earthcenter_point > cos_start_earthcenter_stop:
            d_visible = d
        else:
            break

    return d_visible

In [ ]:
calculate_line_of_sight_endpoint(
    Point(
        lat=radar.lat,
        lon=radar.lon,
        alt=elevationAt(radar.lat, radar.lon),
    ),
    0,
    max_dist * 1000.0,  # [m]
    1,
)

In [ ]:
# Source - https://stackoverflow.com/a/17662363
# Posted by BBSysDyn
# Retrieved 2025-12-11, License - CC BY-SA 4.0

import math, numpy as np

def get_bearing(lat1,lon1,lat2,lon2):
    dLon = lon2 - lon1
    y = math.sin(dLon) * math.cos(lat2)
    x = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dLon)
    brng = np.rad2deg(math.atan2(y, x))
    if brng < 0:
        brng+= 360
    return brng


# Plot DEM

In [ ]:
import plotly.graph_objects as go
import numpy as np

In [ ]:
lat_start = 47.1
lat_end = 47.4
lon_start = 8.4
lon_end = 9.0

lats = np.linspace(lat_start, lat_end, num=3601 // 10)
lons = np.linspace(lon_start, lon_end, num=3601 // 10)

surface = np.empty((len(lats), len(lons)), dtype=np.float32)
points = np.empty((len(lats) * len(lons), 3), dtype=np.float32)

k = 0
for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        surface[i, j] = elevationAt(lat, lon)
        points[k, :] = lat, lon, surface[i, j]
        k += 1

# transformer = Transformer.from_crs("epsg:4326", "epsg:4978", always_xy=True)
surface = surface.clip(min=0.0)
points = points.clip(min=0.0)

In [ ]:
from scipy.spatial import Delaunay

In [ ]:
tri.simplices

In [ ]:
tri = Delaunay(points[:, : 1 + 1])

i, j, k = tri.simplices.T  # vertex indices of triangles

In [ ]:
fig = go.Figure(
    data=[
        go.Mesh3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            i=i.tolist(),
            j=j.tolist(),
            k=k.tolist(),
            lightposition=dict(
                x=1000,
                y=1000,
                z=1000,
            ),
        ),
        go.Scatter3d(
            x=[47.349444],
            y=[8.491389],
            z=[870],  # height value on the surface
            mode="markers",
            marker=dict(size=6, color="red"),
            name="Üetliberg",
        ),
    ]
)

fig.update_layout(
    # margin=dict(l=65, r=50, b=65, t=90),
    scene={
        # "xaxis": {"autorange": "reversed"},
        "yaxis": {"autorange": "reversed"},
    },
)

fig.write_html("surface_mesh.html")

In [ ]:
lats[4], lons[342]

In [ ]:
np.where(surface == surface.max())

In [ ]:
fig = go.Figure(
    data=[
        go.Surface(y=lats, x=lons, z=surface),
        go.Scatter3d(
            y=[47.349444],  # latitude or y-coordinate
            x=[8.491389],  # longitude or x-coordinate
            z=[870],  # height value on the surface
            mode="markers",
            marker=dict(size=6, color="red"),
            name="Üetliberg",
        ),
    ],
)

fig.update_layout(
    # margin=dict(l=65, r=50, b=65, t=90),
    scene={
        "xaxis": {"autorange": "reversed"},
        "yaxis": {"autorange": "reversed"},
    },
)

fig.write_html("surface.html")

In [ ]:
from pyproj import Transformer


transformer = Transformer.from_crs("epsg:4326", "epsg:4978", always_xy=True)

for 

In [ ]:
import pandas as pd

# Read data from a csv
z_data = pd.read_csv(
    "https://raw.githubusercontent.com/plotly/datasets/master/api_docs/mt_bruno_elevation.csv"
)

fig = go.Figure(data=[go.Surface(z=z_data.values)])

fig.update_layout(
    title=dict(text="Mt Bruno Elevation"),
    autosize=False,
    width=500,
    height=500,
    margin=dict(l=65, r=50, b=65, t=90),
)

fig.show()

In [ ]:
# We have to use ray marching to check whether there is a peak between the origin
# and the given direction
def shoot_ray_coverage(
    origin: geopy.Point,
    origin_height: float,
    target_height: float,
    theta: float,
    dist_step,
    max_dist: float,
) -> geopy.Point:
    """
    Shoot a ray from the origin and return the closest point intersected by it.
    """
    R_eff = 4 / 3 * 6371000
    prev_point = origin
    for d in np.arange(dist_step, max_dist + dist_step, dist_step):
        point = burstvincentydistance(origin, d, theta)
        point_height = elevationAt(point.latitude, point.longitude)
        # minus-term: Correct for earth curvature
        point_height_eff = point_height - (d / R_eff) ** 2

        height_line_of_sight = origin_height + d / max_dist * (
            target_height - origin_height
        )
        if point_height_eff >= height_line_of_sight:
            # We have a peak, so the coverage is the last point without peak.
            return prev_point
        else:
            prev_point = point
    return prev_point

In [ ]:
%%timeit
shoot_ray_coverage(
    origin,
    elevationAt(radar.lat, radar.lon),
    target_flight_height,
    0.0,
    dist_step,
    max_dist,
)

In [ ]:
thetas = np.arange(0, 360, theta_res)
points = np.empty((thetas.shape[0], 2))
for i, theta in enumerate(thetas):
    point = shoot_ray_coverage(
        origin,
        origin_height,
        target_flight_height,
        theta,
        dist_step,
        max_dist,
    )
    points[i, :] = point.latitude, point.longitude
points = np.vstack([points, points[0, :]])

In [ ]:
points.min(axis=0), points.max(axis=0)

In [ ]:
import json

In [ ]:
points[:10, :]

In [ ]:
np.stack([points[:, 1], points[:, 0]], axis=1)

In [ ]:
with open("test_file.geosjon", "w") as file:
    json.dump(
        {
            "type": "Polygon",
            "coordinates": [np.stack([points[:, 1], points[:, 0]], axis=1).tolist()],
        },
        file,
    )

In [ ]:
import geopandas as gpd

In [ ]:
df = gpd.read_file("test_file.geosjon")
df_ref = gpd.read_file("reference.geosjon")

polygon_obtained = df.iloc[0, :]["geometry"]
polygon_reference = df_ref.iloc[0, :]["geometry"]

In [ ]:
polygon_reference.difference(polygon_obtained)